# Rede Neural

Implementaremos um modelo de Rede Neural inicial, a fim de buscar uma maneira de classificação automática dos nossos dados mas com seus hiperparâmetros predefinidos e fixos

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, os, random, sklearn
import tensorflow as tf
import scipy.stats as stats
import keras_tuner as kt
from scipy.stats import kurtosis
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tqdm import tqdm

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
# Carrega os dados do Random Forest
X_train_scaled = np.load("dados_treino_teste_best_rf_seed/X_train_scaled_best_rf.npy")
X_test_scaled = np.load("dados_treino_teste_best_rf_seed/X_test_scaled_best_rf.npy")
y_train = np.load("dados_treino_teste_best_rf_seed/y_train_best_rf.npy")
y_test = np.load("dados_treino_teste_best_rf_seed/y_test_best_rf.npy")
nomes_classes = np.load("dados_treino_teste_best_rf_seed/nomes_classes_rf_ms.npy", allow_pickle=True)

print("Dados de treino/teste carregados com sucesso para Redes Neurais.")
print(f"Shape de X_train_scaled: {X_train_scaled.shape}")


In [ ]:
# Cria nosso Modelo de Rede Neural
model = keras.models.Sequential([
    keras.Input(shape=(120,)),
    
    # Adicionamos regularização L2 e aumentamos o Dropout
    keras.layers.Dense(128, activation="relu", kernel_regularizer=l2(0.005)),
    keras.layers.BatchNormalization(), 
    keras.layers.Dropout(0.4), 
    
    keras.layers.Dense(64, activation="relu", kernel_regularizer=l2(0.005)),
    keras.layers.BatchNormalization(), 
    keras.layers.Dropout(0.4),
    
    keras.layers.Dense(6, activation="softmax")
], name="analise_mafaulda_antioverfitting")

In [ ]:
# Compila nosso Modelo, ja lhe dando Funções e Parâmetros
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=0.0005),
              metrics=["accuracy"])

In [ ]:
# Configura a Parada Antecipada
callback_parada = EarlyStopping(
    monitor='val_loss', 
    patience=20,               
    restore_best_weights=True 
)

In [ ]:
# Executa o treinamento passando o callback criado
historico = model.fit(
    X_train_scaled, 
    y_train, 
    epochs=100,                  
    validation_data=(X_test_scaled, y_test),
    callbacks=[callback_parada],
    verbose=1
)

In [ ]:
# Gera nosso Gráfico para visualização da Performance
pd.DataFrame(historico.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1.1)
plt.title('Performance do Modelo Inicial de Rede Neural')
plt.xlabel('Epochs')
plt.ylabel('Métrica')
plt.show()

# Matriz de Confusão para o Modelo Inicial
previsoes_probabilidades = model.predict(X_test_scaled)
y_predito = np.argmax(previsoes_probabilidades, axis=1)
matriz_dados = confusion_matrix(y_test, y_predito)

plt.figure(figsize=(8, 6))
sns.heatmap(
    matriz_dados,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=nomes_classes,
    yticklabels=nomes_classes
)
plt.ylabel("Classe Real")
plt.xlabel("Classe Predita")
plt.title("Matriz de Confusão (Modelo Inicial)")
plt.show()

Partindo do fato de ja termos feito um modelo simplificado de nossa Rede Neural, iremos agora aplicar um modelo de busca de hiperparêmetros através do Keras Tuner, a fim de escolher a rede que terá o melhor desempenho geral, começando pelo "Random Search" e depois pelo "Bayesian Optimization"

In [ ]:
# Definição do Modelo para Keras Tuner
def Rede_Neural_Dinamica(hp):
    model = keras.Sequential(name="analise_mafaulda_antioverfitting_tuning")
    model.add(keras.Input(shape=(X_train_scaled.shape[1],)))

    # Testa de 1 a 4 camadas ocultas
    num_layers = hp.Int("num_layers", min_value=1, max_value=4, step=1)

    for i in range(num_layers):
        # Neurônios por camada
        hp_units = hp.Int(
            f"units_{i}", min_value=32, max_value=256, step=32, default=128
        )

        # Regularização L2 por camada
        hp_l2 = hp.Choice(f"l2_{i}", values=[0.01, 0.005, 0.001], default=0.005)

        model.add(
            layers.Dense(
                units=hp_units, activation="relu", kernel_regularizer=l2(hp_l2)
            )
        )
        model.add(layers.BatchNormalization())

        # Dropout por camada
        hp_dropout = hp.Float(
            f"dropout_{i}", min_value=0.2, max_value=0.5, step=0.1, default=0.4
        )
        model.add(layers.Dropout(rate=hp_dropout))

    # Camada de saída fixa para as 6 classes
    model.add(layers.Dense(len(nomes_classes), activation="softmax"))

    # Testando taxas de aprendizado próximas à 0.0005
    hp_learning_rate = hp.Choice(
        "learning_rate", values=[0.001, 0.0005, 0.0001], default=0.0005
    )

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        metrics=["accuracy"],
    )

    return model

In [ ]:
# Usaremos o RandomSearch para testar as combinações
tuner_rs = kt.RandomSearch(
    Rede_Neural_Dinamica,
    objective="val_accuracy",
    max_trials=15,
    directory="diretorio_tuning",
    project_name="mafaulda_search_rs", 
)

# Configuração do Callback de Parada Antecipada para Keras Tuner
callback_parada_tuner = EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)


In [ ]:
# Executa a busca dos dados escalados com o callback
tuner_rs.search(
    X_train_scaled,
    y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    callbacks=[callback_parada_tuner],
    verbose=1,
)

In [ ]:
# Pega os resultados finais do Random Search
print("\n--- BUSCA Keras Tuner (Random Search) FINALIZADA ---")
best_hp_rs = tuner_rs.get_best_hyperparameters(num_trials=1)[0]
print(f"Melhor número de camadas encontrado (RS): {best_hp_rs.get('num_layers')}")

# Reconstrói o melhor modelo do Random Search
melhor_modelo_rs = tuner_rs.hypermodel.build(best_hp_rs)

In [ ]:
# Retreina o melhor modelo para capturar o "historico"
print("\n--- Treinando o melhor modelo final do Random Search para gerar o histórico ---")
historico_rs = melhor_modelo_rs.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    callbacks=[callback_parada_tuner],
    verbose=1,
)

pd.DataFrame(historico_rs.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1.1)
plt.title('Performance do Melhor Modelo Keras Tuner (Random Search)')
plt.xlabel('Epochs')
plt.ylabel('Métrica')
plt.show()

In [ ]:
# Mudamos de RandomSearch para BayesianOptimization
tuner_bo = kt.BayesianOptimization( 
    Rede_Neural_Dinamica,
    objective="val_accuracy",
    max_trials=20,  
    directory="diretorio_tuning",
    project_name="mafaulda_search_bayesiano", 
)

In [ ]:
# Executa a busca dos dados escalados com o callback
tuner_bo.search(
    X_train_scaled,
    y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    callbacks=[callback_parada_tuner],
    verbose=1,
)

In [ ]:
# Pega os resultados finais do Bayesian Optimization
print("\n--- BUSCA Keras Tuner (Bayesian Optimization) FINALIZADA ---")
best_hp_bo = tuner_bo.get_best_hyperparameters(num_trials=1)[0]
print(f"Melhor número de camadas encontrado (BO): {best_hp_bo.get('num_layers')}")

# Reconstrói o melhor modelo do Bayesian Optimization
melhor_modelo_bo = tuner_bo.hypermodel.build(best_hp_bo)


In [ ]:
# Retreina o melhor modelo para capturar o "historico"
print("\n--- Treinando o melhor modelo final do Bayesian Optimization para gerar o histórico ---")
historico_bo = melhor_modelo_bo.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    validation_data=(X_test_scaled, y_test),
    callbacks=[callback_parada_tuner],
    verbose=1,
)

pd.DataFrame(historico_bo.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1.1)
plt.title('Performance do Melhor Modelo Keras Tuner (Bayesian Optimization)')
plt.xlabel('Epochs')
plt.ylabel('Métrica')
plt.show()


In [ ]:
# O modelo mostra as probabilidades para cada uma das 6 classes
probas = melhor_modelo.predict(X_test_scaled)

# Converte as probabilidades na classe com o maior valor
y_pred = np.argmax(probas, axis=1)

# Cria a matriz de confusão comparando o gabarito com a previsão
cm = confusion_matrix(y_test, y_pred)

# Plota o gráfico de
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,  
    fmt="d",  
    cmap="Blues",  
    xticklabels=[0, 1, 2, 3, 4, 5], 
    yticklabels=[0, 1, 2, 3, 4, 5],
)

plt.ylabel("Classe Real")
plt.xlabel("Classe Predita")
plt.title("Matriz de Confusão")
plt.show()

cap 2 - hands on ml / influencia dos dados não estarem pareados (variaçao de subdivisão nas medidas temporais (2s = 4s?), os dados estarão numa mesma linha? / de onde vem os 250 arq, são novas leituras ou cortes? / leitura da tese de doutorado